## DynBet QuickStart

### 1. 定义一个游戏

In [103]:
from game import Game


class SimpleGame(Game):
    def __init__(self, seq: list[bool]):
        self.seq = seq
        self.index = 0

    def once(self, bet: float) -> float | None:
        # 从列表轮询获取结果，True 表示赢，False 表示输
        won = self.seq[self.index % len(self.seq)]
        self.index += 1
        if won:
            # 赢了，双倍返还
            return bet * 2
        # 输了，啥也没有 (None)


game = SimpleGame([True])  # 一直赢

### 2. 进行游戏

In [ ]:
from session import Session
from vault import Vault
from strategy import Strategy

vault = Vault(1000.0, 10.0)     # 初始余额为 1000，每局游戏初始下注 10
session = Session(game, vault)  # 创建会话
print(vault.money)              # 1000.0
session.run(Strategy())         # 以空策略运行一次
print(vault.money)              # 1010.0

1000.0
1010.0


### 3. 游戏策略

In [105]:
from strategy import Data


# 这是之后要用到的妙妙工具，可以帮我们看见下注额的变化
class Echo(Strategy):
    def __init__(self, strategy: Strategy, f: str = "-> %.2f\n"):
        super().__init__()
        self.strategy = strategy
        self.format = f

    def bet(self, vault: Vault, data: Data) -> Vault:
        bet = self.strategy.bet(vault, data)
        print(self.format % bet.withdrawal, end="")
        return bet

#### 3.1 基础策略

In [106]:
from strategy import Flat

# 平注
session.run(Echo(Flat(20.0)))  # 下注 20 (覆盖初始下注)
print(vault.money)  # 1030.0

-> 20.00
1030.0


In [107]:
from strategy import Ratio

# 按比例下注
session.run(Echo(Ratio(0.5)))  # 下注初始下注的 50%
print(vault.money)  # 1035.0

-> 5.00
1035.0


In [108]:
from strategy import Allin

# 全仓下注
session.run(Echo(Allin()))  # 下注所有资金
print(vault.money)  # 2070.0

-> 1035.00
2070.0


In [109]:
from strategy import Random

# 随机下注
previous = vault.money
bet = session.run(Echo(Random()))  # 在余额范围内随机下注
print(vault.money)

vault.money = 1000.0  # 重置余额

-> 690.88
2760.8773401452004


#### 3.2 组合策略

In [110]:
strategy = Strategy()  # 创建空策略
strategy.apply(Echo(Flat(100.0)))  # 平注 100
strategy.apply(Echo(Ratio(0.5)))  # 缩放 50%
session.run(strategy)
print(vault.money)  # 1050.0

-> 100.00
-> 50.00
1050.0


In [111]:
# 错误用法: 基础策略不会执行 apply 在它之上的策略
session.run(Echo(Flat(100.0)).apply(Echo(Ratio(0.5))))  # 下注额 100 不会变成 50
print(vault.money)  # 1150.0

-> 100.00
1150.0


In [112]:
# 组合: 下注余额的 10%
strategy = Strategy().apply(Echo(Allin())).apply(Echo(Ratio(0.1)))
session.run(strategy)
print(vault.money)  # 1265.0

-> 1150.00
-> 115.00
1265.0


#### 3.3 分支聚合策略

In [ ]:
from strategy import Max, Min


def apply(target: Strategy):
    # 应用三个平注子策略
    target.apply(Echo(Flat(10.0), "Sub 1 -> %.2f\n"))
    target.apply(Echo(Flat(20.0), "Sub 2 -> %.2f\n"))
    target.apply(Echo(Flat(30.0), "Sub 3 -> %.2f\n"))
    return Echo(target, "Max -> %.2f\n")


# 大小比较
strategy = apply(Max())  # 取最大下注额
session.run(strategy)
print(vault.money)  # 1295.0
print()

strategy = apply(Min())  # 取最小下注额
session.run(strategy)
print(vault.money)  # 1305.0

Sub 1 -> 10.00
Sub 2 -> 20.00
Sub 3 -> 30.00
Max -> 30.00
1295.0

Sub 1 -> 10.00
Sub 2 -> 20.00
Sub 3 -> 30.00
Max -> 10.00
1305.0


In [114]:
from strategy import Average

# 取平均下注额
strategy = apply(Average())
session.run(strategy)
print(vault.money)  # 1325.0

Sub 1 -> 10.00
Sub 2 -> 20.00
Sub 3 -> 30.00
Max -> 20.00
1325.0


#### 3.4 分支选择策略

In [ ]:
from strategy import Fork

strategy1 = Strategy()
strategy2 = Strategy()
strategy3 = Strategy()
# 基类: 条件成立则执行 strategy1, 否则依次执行 strategy2 -> strategy3 -> ...
fork = Fork(strategy1).apply(strategy2).apply(strategy3)

##### 3.4.1 普通分支选择策略

In [ ]:
from strategy import Peak, Lowest

game = Game([True, False]) # 赢一次, 输一次
strategy = Strategy()
strategy.apply(Peak())
strategy.apply(Lowest())